# Association Rules Examples
* Package: MLxtend
* Algorithms: Apriori

# Business Problem Understanding

## Case: Discovering Tourist Behavior Through Tour Package Analysis

**1. Background**
<p>
A regional tourism board is collaborating with multiple travel agencies to better understand how tourists experience destinations. Each travel package (tour) consists of a curated set of attractions visited during a trip. Recently, the tourism board has noticed:</p>

    * Increasing competition among travel agencies
    * A need to design more appealing tour packages
    * Interest in bundling attractions more effectively 
    
However, current tour designs rely heavily on intuition rather than data. To support data-driven decision-making, the board has collected detailed records of past tours, including the attractions included in each tour.

**2. Business Problem**
<p>The tourism board wants to answer:</p>
<i>Which attractions are frequently visited together, and how can this information be used to design better tour packages?</i>
<p>Specifically, they are interested in:</p>

    * Identifying popular combinations of attractions 
    * Understanding co-visitation patterns 
    * Designing new tour bundles 
    * Improving cross-promotion among attractions 

**3. Analytical Approach**
<p>The problem can be addressed using association rule mining, a technique commonly used in market basket analysis. In this context:</p>

    * Each tour = a transaction 
    * Each attraction = an item 
    * Goal = identify frequent itemsets and association rules 

In [2]:
# -- Import libraries --
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

# Load the Dataset

<p>The dataset contains records of tours taken by tourists. Each row represents:</p>

    * A tour ID 
    * A list of attractions included in that tour 
    
<p>Transaction Set</p>

|TranID|Attraction_1,Attraction_2,Attraction_3,Attraction_4,Attraction_5,Attraction_6,...|
|---|:---
|0|Boat Tour,Historic Site,Theater,,,|
|1|Historic Site,City Park,Temple,Art Gallery,,
|2|Boat Tour,Museum,Art Gallery,,,
|3|Aquarium,Shopping Center,Food Tour,Botanical Garden,Scenic Train Ride,
|4|Beach,Local Market,Boat Tour,,,
|5|Mountain Hike,Winery,,,,
|...|...
 
<p>Normalized Table Structure:</p>

|Tour_ID|	Attraction
|---|----|
|T001|	Museum
|T001|	Cathedral
|T001|	Old Town
|T002|	Beach
|T002|	Aquarium
|...|...

<p>Each tour may appear in multiple rows (one per attraction).</p>


In [3]:
# -- Import data --
df = pd.read_csv('https://raw.githubusercontent.com/ttchuang/dataset/master/tours.csv')

In [4]:
# -- properties of the dataset --
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   TourID        1000 non-null   int64 
 1   Attraction_1  1000 non-null   object
 2   Attraction_2  1000 non-null   object
 3   Attraction_3  809 non-null    object
 4   Attraction_4  591 non-null    object
 5   Attraction_5  397 non-null    object
 6   Attraction_6  205 non-null    object
dtypes: int64(1), object(6)
memory usage: 54.8+ KB


In [5]:
# -- preview --
df.head()

,TourID,Attraction_1,Attraction_2,Attraction_3,Attraction_4,Attraction_5,Attraction_6
0,0,Boat Tour,Historic Site,Theater,NaN,NaN,NaN
1,1,Historic Site,City Park,Temple,Art Gallery,NaN,NaN
2,2,Boat Tour,Museum,Art Gallery,NaN,NaN,NaN
3,3,Aquarium,Shopping Center,Food Tour,Botanical Garden,Scenic Train Ride,NaN
4,4,Beach,Local Market,Boat Tour,NaN,NaN,NaN


In [6]:
# -- drop TourID or use it as index -
df.drop('TourID',
       axis = 1,
       inplace = True)

## Convert dataset to one-hot key (binary representation)

In [7]:
df

,Attraction_1,Attraction_2,Attraction_3,Attraction_4,Attraction_5,Attraction_6
0,Boat Tour,Historic Site,Theater,NaN,NaN,NaN
1,Historic Site,City Park,Temple,Art Gallery,NaN,NaN
2,Boat Tour,Museum,Art Gallery,NaN,NaN,NaN
3,Aquarium,Shopping Center,Food Tour,Botanical Garden,Scenic Train Ride,NaN
4,Beach,Local Market,Boat Tour,NaN,NaN,NaN
...,...,...,...,...,...,...
995,Food Tour,Theater,Concert,Scenic Train Ride,Museum,NaN
996,Amusement Park,Scenic Train Ride,Zoo,NaN,NaN,NaN
997,Shopping Center,Boat Tour,Scenic Train Ride,Botanical Garden,Aquarium,NaN
998,Mountain Hike,Shopping Center,Botanical Garden,Boat Tour,Historic Site,NaN


In [25]:
# -- 
df_detail = df.stack().reset_index(name='Attraction').drop(['level_1'],axis = 1)
df_detail

,level_0,Attraction
0,0,Boat Tour
1,0,Historic Site
2,0,Theater
3,1,Historic Site
4,1,City Park
...,...,...
3997,998,Botanical Garden
3998,998,Boat Tour
3999,998,Historic Site
4000,999,Concert


In [9]:
df_detail.rename(columns = {'level_0':'TourID'},
                inplace = True)

In [10]:
df_detail

,TourID,Attraction
0,0,Boat Tour
1,0,Historic Site
2,0,Theater
3,1,Historic Site
4,1,City Park
...,...,...
3997,998,Botanical Garden
3998,998,Boat Tour
3999,998,Historic Site
4000,999,Concert


In [11]:
# -- one-hot key encoding --
basket = df_detail.assign(value=1).pivot_table(
    index='TourID',
    columns='Attraction',
    values='value',
    aggfunc='max',
    fill_value=0
)
basket

Attraction,Amusement Park,Aquarium,Art Gallery,Beach,Boat Tour,Botanical Garden,City Park,Concert,Food Tour,Historic Site,Local Market,Mountain Hike,Museum,Scenic Train Ride,Shopping Center,Skyline View,Temple,Theater,Winery,Zoo
TourID,,,,,,,,,,,,,,,,,,,,
0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0
1,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0
2,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
3,0,1,0,0,0,1,0,0,1,0,0,0,0,1,1,0,0,0,0,0
4,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,0,0,0,0,0,1,1,0,0,0,1,1,0,0,0,1,0,0
996,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1
997,0,1,0,0,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0


In [26]:
# -- Product list --
basket.columns

Index(['Amusement Park', 'Aquarium', 'Art Gallery', 'Beach', 'Boat Tour',
       'Botanical Garden', 'City Park', 'Concert', 'Food Tour',
       'Historic Site', 'Local Market', 'Mountain Hike', 'Museum',
       'Scenic Train Ride', 'Shopping Center', 'Skyline View', 'Temple',
       'Theater', 'Winery', 'Zoo'],
      dtype='object', name='Attraction')

In [13]:
# -- Preview --
basket.head()

Attraction,Amusement Park,Aquarium,Art Gallery,Beach,Boat Tour,Botanical Garden,City Park,Concert,Food Tour,Historic Site,Local Market,Mountain Hike,Museum,Scenic Train Ride,Shopping Center,Skyline View,Temple,Theater,Winery,Zoo
TourID,,,,,,,,,,,,,,,,,,,,
0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0
1,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0
2,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
3,0,1,0,0,0,1,0,0,1,0,0,0,0,1,1,0,0,0,0,0
4,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0


In [14]:
# -- Size of the basket --
basket.shape

(1000, 20)

In [15]:
basket[basket.iloc[:,1]>0]

Attraction,Amusement Park,Aquarium,Art Gallery,Beach,Boat Tour,Botanical Garden,City Park,Concert,Food Tour,Historic Site,Local Market,Mountain Hike,Museum,Scenic Train Ride,Shopping Center,Skyline View,Temple,Theater,Winery,Zoo
TourID,,,,,,,,,,,,,,,,,,,,
3,0,1,0,0,0,1,0,0,1,0,0,0,0,1,1,0,0,0,0,0
11,1,1,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0
21,1,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0
29,0,1,1,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0
36,0,1,1,0,0,0,0,1,0,1,0,0,1,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
968,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
974,0,1,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1
984,0,1,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,1,0,0


In [16]:
# -- Convert Unit --
def convert(x):
    if x == 0:
        return 0
    elif x >= 1:
        return 1    

In [17]:
basket = basket.map(convert)
basket

Attraction,Amusement Park,Aquarium,Art Gallery,Beach,Boat Tour,Botanical Garden,City Park,Concert,Food Tour,Historic Site,Local Market,Mountain Hike,Museum,Scenic Train Ride,Shopping Center,Skyline View,Temple,Theater,Winery,Zoo
TourID,,,,,,,,,,,,,,,,,,,,
0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0
1,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0
2,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
3,0,1,0,0,0,1,0,0,1,0,0,0,0,1,1,0,0,0,0,0
4,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,0,0,0,0,0,1,1,0,0,0,1,1,0,0,0,1,0,0
996,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1
997,0,1,0,0,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0


## Frequent Items

In [27]:
# Build up the frequent items
frequent_itemsets = apriori(basket, 
                            min_support=0.01, 
                            use_colnames=True)

c:\Users\offic\Desktop\School\Virtual Enviornment and Py\Zag2\.venv\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [28]:
frequent_itemsets

,support,itemsets
0,0.200,(Amusement Park)
1,0.176,(Aquarium)
2,0.218,(Art Gallery)
3,0.187,(Beach)
4,0.207,(Boat Tour)
...,...,...
302,0.012,"(Temple, Museum, Theater)"
303,0.012,"(Zoo, Temple, Museum)"
304,0.013,"(Zoo, Museum, Theater)"
305,0.011,"(Zoo, Shopping Center, Theater)"


## Create Assciation Rules

In [29]:
# -- Create the rules --
rules = association_rules(frequent_itemsets, 
                          metric="lift", 
                          min_threshold=1)
rules[['antecedents','consequents','support','confidence','lift']]

,antecedents,consequents,support,confidence,lift
0,(Art Gallery),(Amusement Park),0.045,0.206422,1.032110
1,(Amusement Park),(Art Gallery),0.045,0.225000,1.032110
2,(Beach),(Amusement Park),0.038,0.203209,1.016043
3,(Amusement Park),(Beach),0.038,0.190000,1.016043
4,(Amusement Park),(Boat Tour),0.044,0.220000,1.062802
...,...,...,...,...,...
669,"(Zoo, Skyline View)",(Temple),0.012,0.279070,1.291990
670,"(Temple, Skyline View)",(Zoo),0.012,0.279070,1.257071
671,(Zoo),"(Temple, Skyline View)",0.012,0.054054,1.257071
672,(Temple),"(Zoo, Skyline View)",0.012,0.055556,1.291990


## Extract Specific Rules

In [30]:
# -- Filter high lift and confidence vlaues --
rules[ (rules['lift'] >= 6) &
       (rules['confidence'] >= 0.8) ][['antecedents','consequents','support','confidence','lift']]

,antecedents,consequents,support,confidence,lift
